# Seq Len Hidden State Debug


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd

from pfns.experiments.model_benchmarks.model_registry import get_models_from_names
from pfns.experiments.model_benchmarks.seq_len_debug_utils import (
    plot_avg_metric_per_layer_per_head,
    plot_recurrent_metric_per_layer,
    plot_recurrent_metric_per_head,
    run_hidden_state_tracking,
    summarize_hidden_state_by_seqlen,
)
from pfns.utils import get_default_device

plt.rcParams['figure.dpi'] = 400

## Config


In [ ]:
EXPERIMENT = {
    'name': 'seq_len_hidden_state_debug',
    'num_classes': 5,
    'num_features': 10,
    'num_test_samples': 100,
    'num_repetitions': 5,
    'data_generation_seed': 42,
    'seqlen_list': [250, 500, 1_000, 2_000, 4_000, 8_000, 16_000, 32_000, 64_000],
}

MODEL_NAMES = [
    # 'Linear_Attention_FLA_Comb_ST',
    # 'Linear_Attention_Non_Causal',
    'Linear_Attention_Non_Causal_fro_norm',
    # 'Linear_Attention_Comb_ST',
    'Linear_Attention_Comb_ST_fro_norm',
    'Linear_Attention_Causal_fro_norm_from_non_causal',
    # 'Linear_Attention_Comb_ST',
]

TRAINING_CONTEXT_LENGTH = 1_000

# Toggle between the original per-run traces and seq-len violin distributions.
HIDDEN_STATE_PLOT_MODE = 'violin'  # 'violin' or 'individual_runs'
HIDDEN_STATE_DISTRIBUTION_ALPHA = 0.3
HIDDEN_STATE_DISTRIBUTION_WIDTH_FRAC = 0.4

device = str(get_default_device())
models_to_compare = get_models_from_names(MODEL_NAMES)

print(f'Device: {device}')
print(f'Models: {list(models_to_compare)}')
print(f'Repetitions: {EXPERIMENT["num_repetitions"]}')
print(f'Seq lens: {EXPERIMENT["seqlen_list"]}')
print(f'Hidden-state plot mode: {HIDDEN_STATE_PLOT_MODE}')

## Run Hidden-State Tracking


In [ ]:
from pathlib import Path

import pandas as pd

from pfns.experiments.model_benchmarks.hashing import experiment_payload_hash
from pfns.experiments.model_benchmarks.model_registry import functional_model_config

HIDDEN_STATE_CACHE_DIR = Path("hidden_state_cache")
HIDDEN_STATE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

HIDDEN_STATE_CACHE_OVERWRITE = True

hidden_state_cache_key = experiment_payload_hash(
    experiment_payload={
        "experiment": EXPERIMENT,
        "models_to_compare": {
            model_name: functional_model_config(model_config)
            for model_name, model_config in models_to_compare.items()
        },
    }
)
hidden_state_cache_path = (
    HIDDEN_STATE_CACHE_DIR / f"{EXPERIMENT['name']}_{hidden_state_cache_key}.pkl"
)

if hidden_state_cache_path.exists() and not HIDDEN_STATE_CACHE_OVERWRITE:
    hidden_state_df = pd.read_pickle(hidden_state_cache_path)
    print(f"Loaded hidden_state_df from cache: {hidden_state_cache_path}")
else:
    hidden_state_df = run_hidden_state_tracking(
        experiment=EXPERIMENT,
        models_to_compare=models_to_compare,
        device=device,
    )
    hidden_state_df.to_pickle(hidden_state_cache_path)
    print(f"Saved hidden_state_df to cache: {hidden_state_cache_path}")

print("hidden_state_df rows:", len(hidden_state_df))
hidden_state_df.head()


## Summaries and Plots


In [ ]:
summary_df = summarize_hidden_state_by_seqlen(hidden_state_df)

print('summary_df rows:', len(summary_df))
summary_df.head()

### Metric Guide
- `abs_max_mean`: average largest absolute value in each recurrent-state head matrix.
- `frobenius_norm_mean`: per-head Frobenius norm of `recurrent_state` matrices.
- `spectral_norm_mean`: per-head spectral norm (largest singular value) of `recurrent_state` matrices.
- `effective_rank_mean`: DeltaProduct-style effective rank `exp(H(sigma / ||sigma||_1))` of each recurrent-state head matrix.
- `stable_rank_mean`: stable rank `||A||_F^2 / ||A||_2^2` of each recurrent-state head matrix.

In [ ]:
all_tensors = sorted(summary_df['tensor_name'].astype(str).unique().tolist())
print(f'Recurrent-state tensors available: {len(all_tensors)}')
for name in all_tensors:
    print(' ', name)

In [ ]:
shared_plot_kwargs = {
    'plot_mode': HIDDEN_STATE_PLOT_MODE,
    'distribution_alpha': HIDDEN_STATE_DISTRIBUTION_ALPHA,
    'distribution_width_frac': HIDDEN_STATE_DISTRIBUTION_WIDTH_FRAC,
    'log_y': True
}

hidden_state_df_plot = hidden_state_df.copy()

plot_specs = [
    # (
    #     hidden_state_df_plot,
    #     'abs_max',
    #     plot_recurrent_metric_per_head,
    #     {
    #         'title_prefix': 'Recurrent-state abs max vs sequence length',
    #         'training_context_length': TRAINING_CONTEXT_LENGTH,
    #         **shared_plot_kwargs,
    #     },
    # ),
    (
        hidden_state_df_plot,
        'frobenius_norm',
        plot_recurrent_metric_per_head,
        {
            'title_prefix': 'Recurrent-state Frobenius norm vs sequence length',
            'training_context_length': TRAINING_CONTEXT_LENGTH,
            **shared_plot_kwargs,
        },
    ),
    (
        hidden_state_df_plot,
        'spectral_norm',
        plot_recurrent_metric_per_head,
        {
            'title_prefix': 'Recurrent-state spectral norm vs sequence length',
            'training_context_length': TRAINING_CONTEXT_LENGTH,
            **shared_plot_kwargs,
        },
    ),
    (
        hidden_state_df_plot,
        'effective_rank',
        plot_recurrent_metric_per_layer,
        {
            'title_prefix': 'Recurrent-state effective rank vs sequence length',
            'training_context_length': TRAINING_CONTEXT_LENGTH,
            **shared_plot_kwargs,
        },
    ),
    (
        hidden_state_df_plot,
        'stable_rank',
        plot_recurrent_metric_per_layer,
        {
            'title_prefix': 'Recurrent-state stable rank vs sequence length',
            'training_context_length': TRAINING_CONTEXT_LENGTH,
            **shared_plot_kwargs,
        },
    ),
    # (
    #     hidden_state_df_plot,
    #     'kv_state_norm',
    #     plot_recurrent_metric_per_layer,
    #     {
    #         'title_prefix': 'KV-state norm vs sequence length',
    #         'training_context_length': TRAINING_CONTEXT_LENGTH,
    #         **shared_plot_kwargs,
    #     },
    # ),
    # (
    #     hidden_state_df_plot,
    #     'k_sum_norm',
    #     plot_recurrent_metric_per_layer,
    #     {
    #         'title_prefix': 'K-sum norm vs sequence length',
    #         'training_context_length': TRAINING_CONTEXT_LENGTH,
    #         **shared_plot_kwargs,
    #     },
    # ),
    # (
    #     hidden_state_df_plot,
    #     'joint_hidden_state_norm',
    #     plot_recurrent_metric_per_layer,
    #     {
    #         'title_prefix': 'Joint hidden-state norm vs sequence length',
    #         'training_context_length': TRAINING_CONTEXT_LENGTH,
    #         **shared_plot_kwargs,
    #     },
    # ),
    # (
    #     hidden_state_df_plot,
    #     'kv_over_ksum_ratio',
    #     plot_recurrent_metric_per_layer,
    #     {
    #         'title_prefix': 'KV-over-K-sum ratio vs sequence length',
    #         'training_context_length': TRAINING_CONTEXT_LENGTH,
    #         **shared_plot_kwargs,
    #     },
    # ),
]

for plot_df, metric, plot_fn, plot_kwargs in plot_specs:
    if metric not in plot_df.columns:
        continue
    metric_values = pd.to_numeric(plot_df[metric], errors='coerce')
    if not metric_values.notna().any():
        continue
    plot_fn(
        plot_df,
        metric=metric,
        **plot_kwargs,
    )


print('Per-head metric summaries across sequence lengths:')
for metric, title_prefix in [
    ('effective_rank', 'Effective rank per layer'),
    ('stable_rank', 'Stable rank per layer'),
]:
    if metric not in hidden_state_df_plot.columns:
        continue
    metric_values = pd.to_numeric(hidden_state_df_plot[metric], errors='coerce')
    if not metric_values.notna().any():
        continue
    plot_avg_metric_per_layer_per_head(
        hidden_state_df_plot,
        metric=metric,
        title_prefix=title_prefix,
    )
